In [ ]:
import torch
import tensorflow as tf

print("--- PyTorch GPU Check ---")
if torch.cuda.is_available():
    print("PyTorch reports CUDA (GPU) is available.")
    print(f"CUDA Device Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Device Count: {torch.cuda.device_count()}")
else:
    print("PyTorch reports CUDA (GPU) is NOT available.")

print("\n--- TensorFlow GPU Check ---")
if tf.test.is_gpu_available():
    print("TensorFlow reports GPU is available.")
    print("GPU Devices:", tf.config.list_physical_devices('GPU'))
else:
    print("TensorFlow reports GPU is NOT available.")


Change your runtime in GPU.

In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
#STEP 1
!pip uninstall -y scipy
!pip install scipy==1.13.1
!pip install numpy==1.26.4 --force-reinstall

In [ ]:
#STEP 2
!pip install nnunetv2


In [ ]:
#STEP 3
import nnunetv2

In [ ]:
#Step 4: Set nnU-Net Paths
import os

os.environ["nnUNet_raw"] = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/content/nnUNet_results"

os.makedirs("/content/nnUNet_raw", exist_ok=True)
os.makedirs("/content/nnUNet_preprocessed", exist_ok=True)
os.makedirs("/content/nnUNet_results", exist_ok=True)

print("Paths Set Successfully")

In [ ]:
#Step 5: Download Pretrained Glioma Model
!wget -O Dataset002_BRATS19.zip https://zenodo.org/records/11582627/files/Dataset002_BRATS19.zip

In [ ]:
#Step 6: Install Pretrained Model
!nnUNetv2_install_pretrained_model_from_zip Dataset002_BRATS19.zip

In [ ]:
#Step 7: Verify Installation
!find /content/nnUNet_results -name "checkpoint_final.pth"

In [ ]:
!rm -rf /content/input_mri/*
!rm -rf /content/output_masks/*

In [ ]:
#Step 6: Create Input and Output Folders
import os

os.makedirs("/content/input_mri", exist_ok=True)
os.makedirs("/content/output_masks", exist_ok=True)

In [ ]:
#Step 7: Upload MRI File
from google.colab import files

uploaded = files.upload()

In [ ]:
#Step 8: Check Filename
import os

for f in os.listdir():
    if f.endswith(".nii.gz"):
        print(f)

In [ ]:
#Step 9: Rename for nnU-Net

import shutil

shutil.move(
    "JSconverted_output.nii",
    "/content/input_mri/Patient004_0000.nii.gz"
)

In [ ]:
#Step 10: Load Pretrained Model
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
import torch

predictor = nnUNetPredictor(
    tile_step_size=0.5,
    use_gaussian=True,
    use_mirroring=True,
    perform_everything_on_device=True,
    device=torch.device("cuda")
)

predictor.initialize_from_trained_model_folder(
    "/content/nnUNet_results/Dataset002_BRATS19/nnUNetTrainer__nnUNetPlans__3d_fullres",
    use_folds=(0,1,2,3,4),
    checkpoint_name="checkpoint_final.pth"
)

In [ ]:
#Step 11: Run Segmentation
import os
import shutil

input_dir = "/content/input_mri"
base_filename = "Patient004"
mri_gz_expected_path = os.path.join(input_dir, f"{base_filename}_0000.nii.gz")
mri_nii_path = os.path.join(input_dir, f"{base_filename}_0000.nii")

# Check if the file exists as .nii and rename it back to .nii.gz
if os.path.exists(mri_nii_path) and not os.path.exists(mri_gz_expected_path):
    os.rename(mri_nii_path, mri_gz_expected_path)
    print(f"Renamed '{mri_nii_path}' back to '{mri_gz_expected_path}' for nnU-Net compatibility.")

source_file = mri_gz_expected_path

# Check if the base file exists before proceeding
if not os.path.exists(source_file):
    raise FileNotFoundError(f"Source file {source_file} not found. Please ensure Patient004_0000.nii.gz is in {input_dir}.")

# Create the additional 3 channel files by copying the first one
# nnUNet expects input files named PatientID_0000.nii.gz, PatientID_0001.nii.gz, etc.
for i in range(1, 4): # For _0001, _0002, _0003
    dest_file = os.path.join(input_dir, f"{base_filename}_{i:04d}.nii.gz")
    if not os.path.exists(dest_file): # Avoid copying if it already exists
        shutil.copy(source_file, dest_file)
        print(f"Copied {source_file} to {dest_file}")
    else:
        print(f"File {dest_file} already exists, skipping copy.")


predictor.predict_from_files(
    "/content/input_mri",
    "/content/output_masks",
    save_probabilities=False,
    overwrite=True
)

In [ ]:
#STEP 12 Checkig
import nibabel as nib
import numpy as np

mask = nib.load("/content/output_masks/Patient004.nii.gz")
mask_data = mask.get_fdata()

print("Shape:", mask_data.shape)
print("Unique values:", np.unique(mask_data))

# Label | Meaning                                   |
#| ----- | ----------------------------------------- |
#| **0** | Background (Normal Brain Area)            |
#| **1** | Necrotic / Non-enhancing Tumor Core       |
#| **2** | Peritumoral Edema (Swelling around tumor)
#| **4** | Enhancing Tumor                           |


In [ ]:
#Step 13: Check Results Download Output Mask
import os

print(os.listdir("/content/output_masks"))
#Step 13: Download Output Mask
from google.colab import files

files.download("/content/output_masks/Patient004.nii.gz")

In [ ]:
#STEP 14 tumore find in slixces
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np
import os

# Define the paths for the MRI and mask files
mri_gz_path = "/content/input_mri/Patient004_0000.nii.gz"
mask_path = "/content/output_masks/Patient004.nii.gz"

# Temporarily rename the MRI file to remove the .gz extension for nibabel to load it as uncompressed
# This assumes the file content is uncompressed NIfTI despite the .gz extension
# We check if the file exists before attempting to rename
if os.path.exists(mri_gz_path):
    mri_nii_path = mri_gz_path.replace('.nii.gz', '.nii')
    os.rename(mri_gz_path, mri_nii_path)
    print(f"Temporarily renamed '{mri_gz_path}' to '{mri_nii_path}'")
else:
    mri_nii_path = mri_gz_path.replace('.nii.gz', '.nii') # For consistent variable naming even if file not found
    print(f"Warning: Original file '{mri_gz_path}' not found. Attempting to load '{mri_nii_path}'.")


try:
    # Load the MRI and mask data
    mri = nib.load(mri_nii_path).get_fdata()
    mask = nib.load(mask_path).get_fdata()

    print("MRI shape:", mri.shape)
    print("Mask shape:", mask.shape)
    print("Unique mask values:", np.unique(mask))

    # Display slices where tumor (mask value > 0) is found
    for i in range(mask.shape[2]):
        if np.max(mask[:,:,i]) > 0:
            print("Tumor found in slice:", i)

            plt.figure(figsize=(6,6))
            plt.imshow(mri[:,:,i], cmap="gray")
            plt.imshow(mask[:,:,i], cmap="jet", alpha=0.5)
            plt.title(f"Slice {i}")
            plt.axis("off")
            plt.show()

except Exception as e:
    print(f"An error occurred during loading or plotting: {e}")
finally:
    # Rename the MRI file back to its original .nii.gz extension
    # This is important in case subsequent steps expect the .nii.gz name
    if os.path.exists(mri_nii_path):
        os.rename(mri_nii_path, mri_gz_path)
        print(f"Renamed '{mri_nii_path}' back to '{mri_gz_path}'")


In [ ]:
#tep 15: Checking Tumor Voxels in the Segmentation Mask
import numpy as np
import nibabel as nib

mask = nib.load("/content/output_masks/Patient004.nii.gz").get_fdata()

print("Unique values:", np.unique(mask))
print("Non-zero voxels:", np.sum(mask > 0))

In [ ]:
#SSTEP 16 save the result as PNG
import nibabel as nib
import matplotlib.pyplot as plt
import os

mri_gz_path = "/content/input_mri/Patient004_0000.nii.gz"
mask_path = "/content/output_masks/Patient004.nii.gz"

# Temporarily rename the MRI file to remove the .gz extension for nibabel to load it as uncompressed
# This assumes the file content is uncompressed NIfTI despite the .gz extension
if os.path.exists(mri_gz_path):
    mri_nii_path = mri_gz_path.replace('.nii.gz', '.nii')
    os.rename(mri_gz_path, mri_nii_path)
    print(f"Temporarily renamed '{mri_gz_path}' to '{mri_nii_path}'")
else:
    mri_nii_path = mri_gz_path.replace('.nii.gz', '.nii') # For consistent variable naming even if file not found
    print(f"Warning: Original file '{mri_gz_path}' not found. Attempting to load '{mri_nii_path}'.")

try:
    mri = nib.load(mri_nii_path).get_fdata()
    mask = nib.load(mask_path).get_fdata()

    slice_num = 14

    plt.figure(figsize=(8,8))
    plt.imshow(mri[:,:,slice_num], cmap="gray")
    plt.imshow(mask[:,:,slice_num], cmap="jet", alpha=0.5)
    plt.axis("off")

    plt.savefig("/content/glioma_segmentation4.png",
                bbox_inches="tight",
                pad_inches=0)

    plt.show()

except Exception as e:
    print(f"An error occurred during loading or plotting: {e}")
finally:
    # Rename the MRI file back to its original .nii.gz extension
    if os.path.exists(mri_nii_path):
        os.rename(mri_nii_path, mri_gz_path)
        print(f"Renamed '{mri_nii_path}' back to '{mri_gz_path}'")

        #download
from google.colab import files
files.download("/content/glioma_segmentation4.png")

In [ ]:
import os

old_path = "/content/input_mri/Patient004_0000.nii.gz"
new_path = "/content/input_mri/Patient004_0000.nii"

if os.path.exists(old_path):
    os.rename(old_path, new_path)

print("Renamed successfully")

mri = nib.load("/content/input_mri/Patient004_0000.nii").get_fdata()
mask = nib.load("/content/output_masks/Patient004.nii.gz").get_fdata()

In [ ]:
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np

mri = nib.load("/content/input_mri/Patient004_0000.nii").get_fdata()
mask = nib.load("/content/output_masks/Patient004.nii.gz").get_fdata()

# unique tumor slices only
tumor_slices = sorted(set([i for i in range(mask.shape[2]) if np.any(mask[:, :, i] > 0)]))

print("Tumor slices:", tumor_slices)

for slice_idx in tumor_slices:

    plt.figure(figsize=(6,6))
    plt.imshow(mri[:, :, slice_idx], cmap="gray")
    plt.imshow(mask[:, :, slice_idx], cmap="jet", alpha=0.5)

    plt.title(f"Slice {slice_idx}")
    plt.axis("off")
    plt.show()
    plt.close()

In [ ]:
%matplotlib inline

In [ ]:
import os

Segment_folder = "/content/KFtumor_slices"
os.makedirs(Segment_folder, exist_ok=True)


import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np
import os

mri = nib.load("/content/input_mri/Patient004_0000.nii").get_fdata()
mask = nib.load("/content/output_masks/Patient004.nii.gz").get_fdata()

tumor_slices = [i for i in range(mask.shape[2]) if np.any(mask[:, :, i] > 0)]

for slice_no in tumor_slices:

    plt.figure(figsize=(6,6))
    plt.imshow(mri[:, :, slice_no], cmap="gray")
    plt.imshow(mask[:, :, slice_no], cmap="jet", alpha=0.5)
    plt.title(f"Slice {slice_no}")
    plt.axis("off")

    save_path = f"/content/KFtumor_slices/slice_{slice_no}.png"
    plt.savefig(save_path, bbox_inches="tight", dpi=300)

    plt.close()

    import shutil
from google.colab import files

shutil.make_archive("/content/KFtumor_slices", 'zip', "/content/KFtumor_slices")
files.download("/content/KFtumor_slices.zip")

In [ ]:
import shutil

shutil.make_archive(
    "/content/Dataset002_BRATS19",
    "zip",
    "/content/nnUNet_results",
    "Dataset002_BRATS19"
)

In [ ]:
from google.colab import files
files.download("/content/Dataset002_BRATS19.zip")